# Notebook 19 — Memory-Efficient and Distributed Training

    ## Learning objectives

    - Use gradient accumulation without changing gradient scale
- Explain activation checkpointing and mixed precision tradeoffs
- Distinguish data, tensor, pipeline, and fully sharded parallelism

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 19.1 Where memory goes

Training memory includes weights, gradients, optimizer states, saved activations, and
temporary workspaces. Full-precision Adam can require roughly 16 bytes per parameter
before activations, depending on precision/master-weight choices. Sequence length makes
activations especially important. Measure rather than relying only on rules of thumb.


In [ ]:
def rough_gib(params_b, weight_bytes=2, grad_bytes=2, optimizer_bytes=8):
    return params_b * 1e9 * (weight_bytes + grad_bytes + optimizer_bytes) / 2**30
for size in [0.5, 7, 70]:
    print(f"{size:g}B params: {rough_gib(size):.1f} GiB before activations/workspace")


## 19.2 Gradient accumulation

For \(K\) microbatches, divide each microbatch loss by \(K\), call backward K times,
then update once. Effective global batch is
`microbatch × accumulation × data_parallel_world_size`. Token counts—not example counts—
are the more faithful batch unit for variable-length language data.


In [ ]:
import torch
from torch import nn
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
accumulation_steps = 4
opt.zero_grad(set_to_none=True)
for micro_step in range(accumulation_steps):
    x, y = torch.randn(3, 4), torch.randint(0, 2, (3,))
    loss = nn.functional.cross_entropy(model(x), y) / accumulation_steps
    loss.backward()
nn.utils.clip_grad_norm_(model.parameters(), 1.0)
opt.step()
print("one optimizer update from", accumulation_steps, "microbatches")


## 19.3 Checkpointing, precision, and parallelism

Activation checkpointing saves selected inputs and recomputes forward regions during
backward: less memory, more compute. BF16 has FP32-like exponent range and is preferred
on supported hardware; FP16 commonly needs loss scaling. TF32 accelerates selected FP32
matrix operations on NVIDIA hardware.

- **DDP/data parallel:** replica per device; shards batches, synchronizes gradients.
- **FSDP/ZeRO:** shards parameters, gradients, and/or optimizer states.
- **Tensor parallel:** shards operations within layers; communication intensive.
- **Pipeline parallel:** shards layer ranges; introduces bubbles/scheduling complexity.

Hugging Face Accelerate provides a common preparation layer; DeepSpeed and FSDP supply
more aggressive sharding. Distributed correctness comes before scaling benchmarks.


## 19.4 A more complete memory model

Parameter memory includes weights, gradients, optimizer moments, and sometimes FP32 master
weights. Activation memory scales with batch, sequence length, width, layers, attention
implementation, and which tensors backward saves. Temporary buffers, allocator fragmentation,
communication buckets, kernels, and the CUDA context also matter. Peak memory—not steady-state
snapshots—determines whether a step succeeds.

Estimate analytically, then measure `max_memory_allocated` and `max_memory_reserved` after
resetting peak statistics. Reserved memory is the allocator pool and can exceed active tensor
memory. An OOM after several steps may come from variable length, evaluation generation,
leaked graph references, logging tensors without `.item()`, or checkpoint/save spikes.
Sequence length often has nonlinear impact because naive attention intermediates are quadratic.
FlashAttention changes that intermediate-memory term, not all activations.


In [ ]:
# Compare rough parameter-state scenarios.
def state_memory(params, weight=2, grad=2, master=0, moments=8):
    return params * (weight + grad + master + moments)
params = 7_000_000_000
scenarios = {
    "mixed precision AdamW": state_memory(params, 2, 2, 4, 8),
    "bf16 weights+grads, fp32 moments": state_memory(params, 2, 2, 0, 8),
    "weights only inference": state_memory(params, 2, 0, 0, 0),
}
for name, value in scenarios.items(): print(name, f"{value/2**30:.1f} GiB")
print("These exclude activations, buffers, fragmentation, and communication.")


## 19.5 Precision and checkpointing mechanics

FP16 has limited exponent range; gradient scaling multiplies loss before backward, detects
overflow, skips invalid updates, and adjusts scale. BF16 retains FP32's exponent width with
fewer mantissa bits and generally avoids loss scaling, but hardware support matters. FP32
accumulation may still be used inside reductions. TF32 changes eligible NVIDIA matrix
operations while tensors remain FP32. Precision is per-operation policy, not one global dtype.

Activation checkpointing partitions the graph and saves only boundary inputs, replaying
forward operations during backward. It must preserve RNG behavior for dropout and avoid
side effects. More segments save more memory but add recompute and overhead. “Gradient
checkpointing” is unrelated to saving training checkpoints. Combine it with FlashAttention,
accumulation, and length/batch changes only after measuring interactions.


In [ ]:
# Demonstrate checkpointed forward/backward on CPU or GPU.
from torch.utils.checkpoint import checkpoint
device = "cuda" if torch.cuda.is_available() else "cpu"
deep = torch.nn.Sequential(*[
    torch.nn.Sequential(torch.nn.Linear(512, 512), torch.nn.GELU())
    for _ in range(8)]).to(device)
x = torch.randn(16, 512, device=device, requires_grad=True)
def run_segment(start, end, value):
    for layer in list(deep.children())[start:end]: value = layer(value)
    return value
h = checkpoint(lambda z: run_segment(0, 4, z), x, use_reentrant=False)
y = checkpoint(lambda z: run_segment(4, 8, z), h, use_reentrant=False)
y.square().mean().backward()
print("output", y.shape, "input grad norm", x.grad.norm().item())


## 19.6 Distributed strategy decision guide

Use DDP when one model replica plus optimizer fits per GPU and throughput scales with more
data. FSDP/ZeRO shard states when replicas do not fit; sharding stages trade memory for
communication and more complex checkpointing. Tensor parallelism splits layer matrices and
benefits from fast intra-node interconnect. Pipeline parallelism splits layer ranges but has
bubble and microbatch scheduling costs. Sequence/context parallelism divides sequence-related
work. Expert parallelism distributes MoE experts. Large jobs combine dimensions.

Network topology is part of the algorithm. Keep high-frequency tensor-parallel collectives on
fast links; use data parallel across slower nodes where possible. Measure scaling efficiency
against one device, communication time, idle/bubble time, tokens/sec, convergence per token,
and checkpoint duration. A configuration that processes tokens faster but changes effective
batch or numerical behavior is not a controlled speed comparison.

In Colab, focus on single-GPU accumulation, checkpointing, mixed precision, and adapters.
Multi-GPU examples belong on controlled Linux infrastructure with Accelerate configuration
committed alongside the run.


## 19.7 Memory-efficiency reference

| Method | Saves | Costs/constraints |
|---|---|---|
| Smaller microbatch | Activations | More accumulation/less device utilization |
| Shorter sequence | Activations and attention/cache | May truncate necessary evidence |
| BF16/FP16 | Weights/activations/gradients | Numeric/hardware considerations |
| Activation checkpointing | Saved activations | Forward recomputation |
| FlashAttention | Attention intermediates/IO | Backend eligibility |
| LoRA | Trainable gradients/optimizer states | Frozen-base compute remains |
| QLoRA | Frozen base weights | Quantization kernels/quality |
| FSDP/ZeRO | Sharded states | Communication/complex checkpointing |

Apply techniques based on measured dominant memory. Do not stack every optimization blindly: some
interact, introduce unsupported kernels, or reduce throughput more than an alternative batch/length
choice. After an OOM, a notebook runtime may retain tensors in exceptions/history; delete references,
run garbage collection, empty device cache where applicable, or restart before comparing.

Distributed scaling correctness checks include identical data weighting, synchronized updates, no
duplicate/omitted samples, correct metric reductions, checkpoint reload at different world size where
supported, and convergence parity on a small controlled run.


## 19.8 Sharding changes checkpoint semantics

Data parallelism replicates parameters; FSDP and ZeRO shard some combination of parameters, gradients, and optimizer state; tensor and pipeline parallelism partition computation. Consequently a checkpoint may be full, sharded, rank-local, or dependent on a world-size and wrapping plan. Decide how training checkpoints resume and how portable inference artifacts are consolidated. Test resharding and restore before a long run. Record topology, framework versions, mixed-precision policy, and state-dictionary type.


In [ ]:
params=7_000_000_000; bytes_per_param={"weights":2,"grads":2,"adam":8}
for world in (1,2,8): print("world",world,"ideal fully sharded GiB",params*sum(bytes_per_param.values())/world/2**30)


## 19.9 Distributed correctness probes

Run one deterministic update on one device and on the distributed configuration, then compare losses and parameters within justified tolerances. Ensure samplers do not duplicate evaluation examples, global metrics aggregate sums and counts, all ranks agree on step boundaries, and unused-parameter behavior is understood. Watch for hangs caused by divergent control flow or exceptions on one rank. Throughput measurements must report global valid tokens per second and include communication, synchronization, and data loading after warmup.


In [ ]:
rank_sums=[torch.tensor(12.),torch.tensor(9.)]; rank_counts=[torch.tensor(4),torch.tensor(3)]
global_mean=sum(rank_sums)/sum(rank_counts); wrong=sum(s/c for s,c in zip(rank_sums,rank_counts))/len(rank_sums)
print("global",global_mean.item(),"mean of rank means",wrong.item())


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [PyTorch FSDP](https://docs.pytorch.org/docs/stable/fsdp.html)
- [PyTorch activation checkpointing](https://docs.pytorch.org/docs/stable/checkpoint.html)
- [ZeRO](https://arxiv.org/abs/1910.02054)


## Exercises

    1. Prove accumulated gradients match one large batch when example weighting is identical.
2. Checkpoint alternating layers and measure memory versus step time.
3. Draw a topology for training a model that cannot fit on one node.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
